# SonarScan AI: Model Training (Google Colab)

**PS 26057 (MoES / NIOT): AI-Powered Underwater Marine Debris & Anomaly Detection from Side-Scan Sonar**

This notebook trains the **two models** used by the FastAPI inference microservice:

| # | Model | Task | Architecture | Data |
|---|-------|------|--------------|------|
| 1 | `sonar_detector` | Multi-class anomaly **detection** (boxes) | YOLO11s | Roboflow sonar_detect + MILCO/NOMBO (figshare 24574879) + AquaScan-1K + AI4Shipwrecks (tiled) |
| 2 | `shipwreck_segmenter` | Pixel-level **segmentation** of wrecks | YOLO11s-seg | AI4Shipwrecks (tiled at native resolution) |

### Unified class taxonomy (detector)

| id | class | source labels | hazard? |
|----|-------|---------------|---------|
| 0 | `shipwreck` | Roboflow `shipwreck`, AI4Shipwrecks masks | yes |
| 1 | `aircraft_wreck` | Roboflow `aircraft` | yes |
| 2 | `mine_like_object` | figshare `MILCO` (cylinders / mine-like contacts) | yes |
| 3 | `debris` | figshare `NOMBO` (non-mine bottom objects), Roboflow `other` (tyres, reef blocks, etc.) | yes |
| 4 | `human_body` | AquaScan-1K `Human` (drowning-victim search) | yes (SAR) |
| 5 | `marine_life` | Roboflow `fish` | **no**: natural, used for false-positive suppression |

### Key data decisions (from the dataset audit)
* **Grayscale everywhere.** Sonar return is single-channel; the gold/brown/grey palettes are display artefacts. Converting all sources to grayscale removes the palette domain gap. The inference service must do the same.
* **AI4Shipwrecks is tiled (1024 px, 25% overlap) at native resolution.** The images are 1728 px wide and up to 18,000 px tall with wrecks covering ~1.5% of pixels. The previous baseline resized whole images to 1024, which shrank the wrecks to a few pixels and capped mAP50 at ~0.24.
* **Leakage-safe splits.** AI4 is split by wreck site, MILCO/NOMBO by blocks of consecutive overlapping tiles, AquaScan by recording session (date + hour), and Roboflow by its given splits with near-duplicate removal.
* **Speckle / ping-dropout augmentation.** Extra training copies with multiplicative gamma speckle and blacked-out ping rows simulate high speckle noise and heave/pitch/roll data dropouts.
* `my_archive` (Kaggle copy) is a duplicate of `archive` and is not used. One corrupted AquaScan PNG is dropped during packaging.

### How to run
1. Upload `sonar_datasets.zip` (made by `ai/tools/package_datasets.py`) to Google Drive at `MyDrive/ISIH2026/sonar_datasets.zip`.
2. `Runtime → Change runtime type → GPU` (T4 works; L4/A100 is faster).
3. Run all cells. Checkpoints are saved to Drive, so if Colab disconnects, re-run everything and training **resumes**.
4. The last cell downloads `sonar_models.zip`. Unzip it into `ai/models/` of the project.

## 1. Environment

In [ ]:
!nvidia-smi -L || echo "No GPU found. Switch the runtime to GPU."
%pip -q install -U ultralytics onnx onnxslim onnxruntime

## 2. Configuration
Set `QUICK_TEST = True` for a 2-epoch smoke test of the whole pipeline (~10 min) before the full run.

In [ ]:
import os, sys, json, re, math, random, shutil, time, zipfile
from pathlib import Path
from collections import Counter, defaultdict

import cv2
import numpy as np

IN_COLAB = "google.colab" in sys.modules

if IN_COLAB:
    from google.colab import drive
    drive.mount("/content/drive")
    DRIVE_ROOT = Path("/content/drive/MyDrive/ISIH2026")
    DATA_ZIP   = DRIVE_ROOT / "sonar_datasets.zip"   # uploaded dataset package
    RUNS_DIR   = DRIVE_ROOT / "runs"                  # checkpoints survive disconnects
    CONTENT    = Path("/content")
else:  # local run (for debugging the data pipeline)
    DRIVE_ROOT = Path(os.environ.get("SONAR_ROOT", "."))
    DATA_ZIP   = DRIVE_ROOT / "sonar_datasets.zip"
    RUNS_DIR   = DRIVE_ROOT / "runs"
    CONTENT    = DRIVE_ROOT / "content"

RAW  = CONTENT / "sonar_datasets"   # unzipped raw data
WORK = CONTENT / "work"             # generated YOLO datasets
DET  = WORK / "det"                 # detection dataset
SEG  = WORK / "seg"                 # segmentation dataset
RUNS_DIR.mkdir(parents=True, exist_ok=True)

# ---- what to run ----
QUICK_TEST      = False    # True -> 2 epochs on 20% of the data, to test the pipeline
TRAIN_DETECTOR  = True
TRAIN_SEGMENTER = True
RESUME          = True     # resume from Drive checkpoints if they exist
REBUILD_DATA    = False    # force regeneration of WORK datasets

# ---- models ----
DET_WEIGHTS = "yolo11s.pt"       # yolo11n.pt = fastest edge model, yolo11m.pt = more accurate
SEG_WEIGHTS = "yolo11s-seg.pt"
DET_NAME    = "sonar_detector"
SEG_NAME    = "shipwreck_segmenter"
IMGSZ       = 1024
EPOCHS      = 2 if QUICK_TEST else 150
PATIENCE    = 30
BATCH       = 8                  # T4 16 GB: 8 for 's' at 1024. A100: 16-24
WORKERS     = 4 if IN_COLAB else 0
FRACTION    = 0.2 if QUICK_TEST else 1.0
SEED        = 42

# ---- data building ----
TILE             = 1024   # AI4Shipwrecks tile size (native pixels)
TILE_OVERLAP     = 0.25
NEG_TILE_RATIO   = 0.5    # negative (no-wreck) tiles kept per positive tile
TERRAIN_NEG_MAX  = 150    # hard-negative tiles from rocky/rough terrain (extras/terrain)
MN_NEG_RATIO     = 1.0    # empty MILCO/NOMBO tiles kept per positive tile
SPECKLE_FRACTION = 0.15   # extra speckle/dropout-augmented copies of train images

CLASSES = ["shipwreck", "aircraft_wreck", "mine_like_object", "debris", "human_body", "marine_life"]
HAZARD  = {"shipwreck": True, "aircraft_wreck": True, "mine_like_object": True,
           "debris": True, "human_body": True, "marine_life": False}

random.seed(SEED); np.random.seed(SEED)
print("RAW:", RAW, "| WORK:", WORK, "| RUNS:", RUNS_DIR)

## 3. Unpack the dataset package

In [ ]:
if not RAW.exists():
    assert DATA_ZIP.exists(), f"Upload sonar_datasets.zip to {DATA_ZIP}"
    t = time.time()
    with zipfile.ZipFile(DATA_ZIP) as zf:
        zf.extractall(CONTENT)
    print(f"Extracted in {time.time() - t:.0f}s")
for d in sorted(RAW.iterdir()):
    print(f"{d.name:16s}", sum(1 for _ in d.rglob("*") if _.is_file()), "files")

## 4. Helpers
YOLO label I/O, grayscale loading, leakage-safe group splitting, and the speckle / ping-dropout augmentation.

In [ ]:
SPLITS = ["train", "val", "test"]

def read_yolo_boxes(txt):
    # Read YOLO box labels, clip them to the image and drop degenerate boxes.
    rows = []
    txt = Path(txt)
    if not txt.exists():
        return rows
    for line in txt.read_text().splitlines():
        v = line.split()
        if len(v) != 5:
            continue
        c, x, y, w, h = int(v[0]), *map(float, v[1:])
        x1, y1 = max(0.0, x - w / 2), max(0.0, y - h / 2)
        x2, y2 = min(1.0, x + w / 2), min(1.0, y + h / 2)
        if x2 - x1 < 0.002 or y2 - y1 < 0.002:
            continue
        rows.append((c, (x1 + x2) / 2, (y1 + y2) / 2, x2 - x1, y2 - y1))
    return rows

def load_gray(path):
    img = cv2.imread(str(path), cv2.IMREAD_GRAYSCALE)
    if img is None:
        raise IOError(f"Cannot read {path}")
    return img

def box_lines(rows):
    return "".join(f"{c} {x:.6f} {y:.6f} {w:.6f} {h:.6f}\n" for c, x, y, w, h in rows)

def save_sample(root, split, stem, img, label_text):
    (root / "images" / split).mkdir(parents=True, exist_ok=True)
    (root / "labels" / split).mkdir(parents=True, exist_ok=True)
    cv2.imwrite(str(root / "images" / split / f"{stem}.jpg"), img, [cv2.IMWRITE_JPEG_QUALITY, 95])
    (root / "labels" / split / f"{stem}.txt").write_text(label_text)

def group_split(groups, weights=None, ratios=(0.8, 0.1, 0.1), seed=SEED):
    # Assign whole groups to train/val/test so near-identical images never cross splits.
    # Greedy: biggest groups first, each goes to the split furthest below its target share.
    weights = weights or [1.0] * len(groups)
    size = defaultdict(float)
    for g, w in zip(groups, weights):
        size[g] += w
    order = sorted(size, key=lambda g: (-size[g], random.Random(f"{seed}{g}").random()))
    total = sum(size.values())
    target = [r * total for r in ratios]
    acc, assign = [0.0, 0.0, 0.0], {}
    for g in order:
        k = max(range(3), key=lambda i: (target[i] - acc[i]) / target[i] if target[i] > 0 else -1e9)
        assign[g] = k
        acc[k] += size[g]
    return [SPLITS[assign[g]] for g in groups]

def speckle_dropout(img, rng):
    # Multiplicative gamma speckle (L looks) + blacked-out ping rows (vehicle motion dropouts).
    f = img.astype(np.float32)
    looks = rng.choice([1, 2, 4, 8])
    f *= rng.gamma(looks, 1.0 / looks, size=f.shape).astype(np.float32)
    if rng.random() < 0.6:
        for _ in range(rng.integers(1, 6)):
            y = rng.integers(0, f.shape[0])
            h = rng.integers(1, max(2, f.shape[0] // 40))
            f[y:y + h] *= 0.0 if rng.random() < 0.7 else 0.3
    return np.clip(f, 0, 255).astype(np.uint8)

def ahash(img):
    small = cv2.resize(img, (8, 8), interpolation=cv2.INTER_AREA)
    return (small > small.mean()).tobytes()

STATS = defaultdict(lambda: defaultdict(lambda: defaultdict(Counter)))  # [dataset][source][split]
def record(dataset, source, split, class_ids):
    STATS[dataset][source][split]["images"] += 1
    if not class_ids:
        STATS[dataset][source][split]["background"] += 1
    for c in class_ids:
        STATS[dataset][source][split][CLASSES[c]] += 1

## 5. Build the detection dataset
### 5.1 Roboflow `sonar_detect` (aircraft / fish / other / shipwreck)
Mixed-sensor sonar images (side-scan, multibeam renders). Given splits are kept; val/test images that are near-duplicates of train images are removed.

In [ ]:
DATA_READY = (DET / "data.yaml").exists() and (SEG / "data.yaml").exists() and not REBUILD_DATA
if not DATA_READY:
    shutil.rmtree(WORK, ignore_errors=True)
    STATS.clear()

    RF_MAP = {0: 1, 1: 5, 2: 3, 3: 0}   # aircraft, fish, other, shipwreck -> unified ids
    rf_root = RAW / "roboflow_sonar"
    train_hashes, dropped = set(), 0
    for src_split, dst_split in [("train", "train"), ("valid", "val"), ("test", "test")]:
        for img_path in sorted((rf_root / src_split / "images").glob("*.jpg")):
            img = load_gray(img_path)
            h = ahash(img)
            if dst_split == "train":
                train_hashes.add(h)
            elif h in train_hashes:
                dropped += 1
                continue
            rows = [(RF_MAP[c], x, y, w, h_) for c, x, y, w, h_ in
                    read_yolo_boxes(rf_root / src_split / "labels" / f"{img_path.stem}.txt")]
            stem = "rf_" + img_path.stem.split("_jpg")[0]
            save_sample(DET, dst_split, stem, img, box_lines(rows))
            record("det", "roboflow", dst_split, [r[0] for r in rows])
    print("Roboflow near-duplicates removed from val/test:", dropped)

### 5.2 MILCO / NOMBO: mine-like objects (figshare 24574879)
Real side-scan survey tiles (2010-2021). Tiles overlap by 20%, so consecutive tiles are grouped (blocks of 20) before splitting. Empty tiles are kept as background at a 1:1 ratio.

In [ ]:
if not DATA_READY:
    MN_MAP = {0: 2, 1: 3}   # MILCO -> mine_like_object, NOMBO -> debris
    items = []
    for year_dir in sorted(p for p in (RAW / "milco_nombo").iterdir() if p.is_dir()):
        for img_path in sorted(year_dir.glob("*.jpg")):
            rows = [(MN_MAP[c], *r) for c, *r in read_yolo_boxes(img_path.with_suffix(".txt"))]
            idx = int(re.match(r"(\d+)", img_path.stem).group(1))
            items.append((img_path, rows, f"{year_dir.name}_{idx // 20}"))

    splits = group_split([g for *_, g in items], weights=[1 + 3 * len(r) for _, r, _ in items])
    rng = random.Random(SEED)
    for split in SPLITS:
        pos = [(p, r) for (p, r, _), s in zip(items, splits) if s == split and r]
        neg = [(p, r) for (p, r, _), s in zip(items, splits) if s == split and not r]
        neg = rng.sample(neg, min(len(neg), int(len(pos) * MN_NEG_RATIO)))
        for img_path, rows in pos + neg:
            save_sample(DET, split, f"mn_{img_path.stem}", load_gray(img_path), box_lines(rows))
            record("det", "milco_nombo", split, [r[0] for r in rows])
    print("MILCO/NOMBO done")

### 5.3 AquaScan-1K: human body / drowning-victim search
Screenshots from consumer side-scan units (1920x1080). Objects are tiny (~4% of width), which is why the detector trains at 1024 px. Screenshots from the same session are near-identical, so they are grouped by recording date + hour.

In [ ]:
def aquascan_session(stem):
    name = stem.split("-", 1)[1] if "-" in stem else stem   # drop Label Studio hash prefix
    for pat in [r"(\d{4}-\d{2}-\d{2})[_ .-](\d{2})", r"(\d{2}[A-Z]{3}\d{2})_(\d{2})"]:
        m = re.search(pat, name)
        if m:
            return "_".join(m.groups())
    return name

if not DATA_READY:
    aq = RAW / "aquascan"
    imgs = sorted((aq / "images").glob("*.jpg"))
    groups = [aquascan_session(p.stem) for p in imgs]
    print("AquaScan sessions:", len(set(groups)))
    for img_path, split in zip(imgs, group_split(groups)):
        rows = [(4, *r) for _, *r in read_yolo_boxes(aq / "labels" / f"{img_path.stem}.txt")]
        save_sample(DET, split, f"aq_{img_path.stem[:40]}", load_gray(img_path), box_lines(rows))
        record("det", "aquascan", split, [r[0] for r in rows])

### 5.4 AI4Shipwrecks → tiles for both the segmenter and the detector
* Train images are split into train/val **by wreck site**; the official test set (different wrecks) stays test.
* Each 1024x1024 tile gets polygon labels (segmenter) and box labels (detector). Mask fragments of one wreck are merged into one box by dilating the mask before finding components.
* Negative tiles plus hard-negative tiles from `extras/terrain` (rocks, reefs, ridges) teach the models what is **not** a wreck.

In [ ]:
def tile_origins(length, tile=TILE, overlap=TILE_OVERLAP):
    if length <= tile:
        return [0]
    stride = int(tile * (1 - overlap))
    xs = list(range(0, length - tile + 1, stride))
    if xs[-1] != length - tile:
        xs.append(length - tile)
    return xs

def mask_to_polygons(mask):
    h, w = mask.shape
    contours, _ = cv2.findContours(mask.astype(np.uint8), cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)
    lines = []
    for c in contours:
        if cv2.contourArea(c) < 30:
            continue
        c = cv2.approxPolyDP(c, 1.0, True).reshape(-1, 2)
        if len(c) < 3:
            continue
        pts = " ".join(f"{x / w:.6f} {y / h:.6f}" for x, y in c)
        lines.append(f"0 {pts}\n")
    return lines

def mask_to_boxes(mask, cls=0, merge_px=25, min_px=8):
    h, w = mask.shape
    merged = cv2.dilate(mask.astype(np.uint8), np.ones((merge_px, merge_px), np.uint8))
    n, labels = cv2.connectedComponents(merged)
    rows = []
    for i in range(1, n):
        ys, xs = np.where((labels == i) & mask)
        if len(xs) == 0:
            continue
        x1, x2, y1, y2 = xs.min(), xs.max() + 1, ys.min(), ys.max() + 1
        if x2 - x1 < min_px or y2 - y1 < min_px:
            continue
        rows.append((cls, (x1 + x2) / 2 / w, (y1 + y2) / 2 / h, (x2 - x1) / w, (y2 - y1) / h))
    return rows

def tile_image(img, mask):
    H, W = img.shape
    for y in tile_origins(H):
        for x in tile_origins(W):
            yield x, y, img[y:y + TILE, x:x + TILE], mask[y:y + TILE, x:x + TILE]

if not DATA_READY:
    ai4 = RAW / "ai4shipwrecks"
    train_imgs = sorted((ai4 / "train" / "images").glob("*.png"))
    site = lambda p: p.stem.rsplit("_", 1)[0]
    ai4_split = dict(zip(train_imgs, group_split([site(p) for p in train_imgs], ratios=(0.8, 0.2, 0.0))))
    ai4_split.update({p: "test" for p in sorted((ai4 / "test" / "images").glob("*.png"))})
    print("AI4 val sites:", sorted({site(p) for p, s in ai4_split.items() if s == "val"}))

    rng = random.Random(SEED)
    for split in SPLITS:
        pos, neg = [], []
        for img_path in [p for p, s in ai4_split.items() if s == split]:
            img = load_gray(img_path)
            mask = load_gray(img_path.parent.parent / "labels" / img_path.name) > 0
            for x, y, t_img, t_mask in tile_image(img, mask):
                stem = f"ai4_{img_path.stem}_{y}_{x}"
                (pos if t_mask.sum() >= 50 else neg).append((stem, t_img, t_mask))
        neg = rng.sample(neg, min(len(neg), int(len(pos) * NEG_TILE_RATIO)))
        for stem, t_img, t_mask in pos + neg:
            has = t_mask.sum() >= 50
            save_sample(SEG, split, stem, t_img, "".join(mask_to_polygons(t_mask)) if has else "")
            rows = mask_to_boxes(t_mask) if has else []
            save_sample(DET, split, stem, t_img, box_lines(rows))
            record("seg", "ai4shipwrecks", split, [0] * len(mask_to_polygons(t_mask)) if has else [])
            record("det", "ai4shipwrecks", split, [r[0] for r in rows])
        print(f"AI4 {split}: {len(pos)} wreck tiles, {len(neg)} background tiles")

    # Hard negatives: rough natural terrain (train only).
    terrain = []
    for img_path in sorted((ai4 / "extras" / "terrain" / "images").glob("*.png")):
        img = load_gray(img_path)
        empty = np.zeros_like(img, dtype=bool)
        terrain += [(f"ai4t_{img_path.stem}_{y}_{x}", t) for x, y, t, _ in tile_image(img, empty)]
    for stem, t_img in rng.sample(terrain, min(len(terrain), TERRAIN_NEG_MAX)):
        save_sample(SEG, "train", stem, t_img, "")
        save_sample(DET, "train", stem, t_img, "")
        record("seg", "terrain_neg", "train", []); record("det", "terrain_neg", "train", [])

### 5.5 Speckle / ping-dropout augmented copies (train only), then the dataset YAMLs

In [ ]:
import yaml

def add_speckle_copies(root, dataset):
    rng = np.random.default_rng(SEED)
    imgs = sorted((root / "images" / "train").glob("*.jpg"))
    picks = rng.choice(len(imgs), size=int(len(imgs) * SPECKLE_FRACTION), replace=False)
    for i in picks:
        p = imgs[i]
        label = (root / "labels" / "train" / f"{p.stem}.txt").read_text()
        save_sample(root, "train", f"spk_{p.stem}", speckle_dropout(load_gray(p), rng), label)
        STATS[dataset]["speckle_aug"]["train"]["images"] += 1
    print(f"{dataset}: +{len(picks)} speckle/dropout copies")

def write_yaml(root, names):
    cfg = {"path": str(root.resolve()), "train": "images/train", "val": "images/val",
           "test": "images/test", "names": dict(enumerate(names))}
    (root / "data.yaml").write_text(yaml.safe_dump(cfg, sort_keys=False))

if not DATA_READY:
    add_speckle_copies(DET, "det")
    add_speckle_copies(SEG, "seg")
    write_yaml(DET, CLASSES)
    write_yaml(SEG, ["shipwreck"])
    json.dump({d: {s: {sp: dict(c) for sp, c in v.items()} for s, v in src.items()} for d, src in STATS.items()},
              open(WORK / "dataset_stats.json", "w"), indent=2)

print(Path(DET / "data.yaml").read_text())

### 5.6 Dataset summary & visual check

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt

stats = json.load(open(WORK / "dataset_stats.json"))
for dataset, sources in stats.items():
    rows = [{"source": src, "split": sp, **cnt} for src, splits in sources.items() for sp, cnt in splits.items()]
    df = pd.DataFrame(rows).fillna(0).set_index(["source", "split"]).astype(int)
    print(f"\n=== {dataset.upper()} ===")
    display(df)
    display(df.groupby(level="split").sum())

def show_grid(root, split="train", n=8, seg=False):
    imgs = sorted((root / "images" / split).glob("*.jpg"))
    imgs = [p for p in imgs if (root / "labels" / split / f"{p.stem}.txt").read_text().strip()]
    picks = random.Random(1).sample(imgs, min(n, len(imgs)))
    fig, axes = plt.subplots(2, 4, figsize=(20, 10))
    colors = plt.cm.tab10.colors
    for ax, p in zip(axes.ravel(), picks):
        img = cv2.cvtColor(load_gray(p), cv2.COLOR_GRAY2RGB)
        h, w = img.shape[:2]
        for line in (root / "labels" / split / f"{p.stem}.txt").read_text().splitlines():
            v = list(map(float, line.split()))
            col = tuple(int(255 * c) for c in colors[int(v[0])])
            if seg:
                pts = (np.array(v[1:]).reshape(-1, 2) * [w, h]).astype(np.int32)
                cv2.polylines(img, [pts], True, col, 3)
            else:
                x, y, bw, bh = v[1:]
                cv2.rectangle(img, (int((x - bw / 2) * w), int((y - bh / 2) * h)),
                              (int((x + bw / 2) * w), int((y + bh / 2) * h)), col, 3)
                cv2.putText(img, CLASSES[int(v[0])], (int((x - bw / 2) * w), max(20, int((y - bh / 2) * h) - 6)),
                            cv2.FONT_HERSHEY_SIMPLEX, 0.9, col, 2)
        ax.imshow(img); ax.set_title(p.stem[:40], fontsize=9); ax.axis("off")
    plt.tight_layout(); plt.show()

show_grid(DET)
show_grid(SEG, seg=True)

## 6. Training
Checkpoints are written straight to Google Drive. If the runtime disconnects, re-run all cells and training resumes from `last.pt`.

Augmentation notes: hue/saturation jitter is off (inputs are grayscale); vertical and horizontal flips are on (port/starboard and along-track symmetry); rotation stays small because side-scan geometry is not rotation-invariant.

In [ ]:
from ultralytics import YOLO

COMMON = dict(imgsz=IMGSZ, epochs=EPOCHS, patience=PATIENCE, batch=BATCH, workers=WORKERS,
              fraction=FRACTION, seed=SEED, cos_lr=True, close_mosaic=15, plots=True, exist_ok=True,
              project=str(RUNS_DIR), hsv_h=0.0, hsv_s=0.0, hsv_v=0.4, fliplr=0.5, flipud=0.5,
              degrees=5.0, translate=0.1, scale=0.5, mosaic=1.0, mixup=0.1)

def train(name, weights, data, **extra):
    last = RUNS_DIR / name / "weights" / "last.pt"
    if RESUME and last.exists():
        try:
            print(f"Resuming {name} from {last}")
            YOLO(str(last)).train(resume=True)
        except Exception as e:  # already finished / incompatible -> keep existing weights
            print(f"Resume skipped: {e}")
    else:
        YOLO(weights).train(data=str(data / "data.yaml"), name=name, **COMMON, **extra)
    best = RUNS_DIR / name / "weights" / "best.pt"
    assert best.exists(), f"No best.pt for {name}"
    return best

if TRAIN_DETECTOR:
    DET_BEST = train(DET_NAME, DET_WEIGHTS, DET)

In [ ]:
if TRAIN_SEGMENTER:
    SEG_BEST = train(SEG_NAME, SEG_WEIGHTS, SEG, copy_paste=0.3)

## 7. Evaluation on the held-out test split + confidence calibration
* **Test metrics** (never seen during training or model selection) go into the model card, so the dashboard shows real numbers.
* **Per-class confidence thresholds** are the confidence values that maximise F1 on the **validation** split. The inference service uses them to suppress low-confidence false positives per class.

In [ ]:
import pandas as pd

def per_class_table(metric, names):
    rows = []
    for i, c in enumerate(metric.ap_class_index):
        rows.append({"class": names[int(c)], "precision": metric.p[i], "recall": metric.r[i],
                     "mAP50": metric.ap50[i], "mAP50-95": metric.ap[i]})
    rows.append({"class": "ALL", "precision": metric.mp, "recall": metric.mr,
                 "mAP50": metric.map50, "mAP50-95": metric.map})
    return pd.DataFrame(rows).round(4)

def best_f1_thresholds(metric, names, floor=0.05):
    px, f1 = getattr(metric, "px", None), getattr(metric, "f1_curve", None)
    out = {n: {"conf": 0.25, "f1": None} for n in names.values()}
    if px is None or f1 is None or len(f1) == 0:
        return out
    for i, c in enumerate(metric.ap_class_index):
        j = int(np.argmax(f1[i]))
        if f1[i][j] <= 0:   # class never detected on val -> keep the default
            continue
        out[names[int(c)]] = {"conf": round(max(float(px[j]), floor), 3), "f1": round(float(f1[i][j]), 4)}
    return out

def evaluate(name, best, data, seg=False):
    model = YOLO(str(best))
    kw = dict(data=str(data / "data.yaml"), imgsz=IMGSZ, batch=BATCH, project=str(RUNS_DIR), exist_ok=True)
    val = model.val(split="val", name=f"{name}_val", plots=False, **kw)
    test = model.val(split="test", name=f"{name}_test", plots=True, **kw)
    names = model.names
    report = {"box": per_class_table(test.box, names)}
    if seg:
        report["mask"] = per_class_table(test.seg, names)
    for k, df in report.items():
        print(f"\n{name}: TEST {k} metrics"); display(df)
    thresholds = best_f1_thresholds(val.seg if seg else val.box, names)
    print("Calibrated thresholds (val):", thresholds)
    return {k: df.to_dict(orient="records") for k, df in report.items()}, thresholds, test.speed

RESULTS = {}
if TRAIN_DETECTOR:
    RESULTS[DET_NAME] = evaluate(DET_NAME, DET_BEST, DET)
if TRAIN_SEGMENTER:
    RESULTS[SEG_NAME] = evaluate(SEG_NAME, SEG_BEST, SEG, seg=True)

## 8. Tiled (sliced) inference on a full-length sonar strip
Full survey strips are far larger than the model input, so inference slides a 1024 px window with 25% overlap, runs both models, and merges overlapping detections with NMS. The FastAPI service will use the same logic.

In [ ]:
import torch, torchvision

def sliced_predict(model, gray, conf=0.25, iou=0.5, tile=TILE, overlap=TILE_OVERLAP):
    H, W = gray.shape
    boxes, scores, classes = [], [], []
    for y in tile_origins(H, tile, overlap):
        for x in tile_origins(W, tile, overlap):
            crop = cv2.cvtColor(gray[y:y + tile, x:x + tile], cv2.COLOR_GRAY2BGR)
            r = model.predict(crop, imgsz=IMGSZ, conf=conf, verbose=False)[0]
            if r.boxes is None or len(r.boxes) == 0:
                continue
            b = r.boxes.xyxy.cpu() + torch.tensor([x, y, x, y])
            boxes.append(b); scores.append(r.boxes.conf.cpu()); classes.append(r.boxes.cls.cpu())
    if not boxes:
        return np.zeros((0, 4)), np.zeros(0), np.zeros(0)
    b, s, c = torch.cat(boxes), torch.cat(scores), torch.cat(classes)
    keep = torchvision.ops.batched_nms(b, s, c, iou)
    return b[keep].numpy(), s[keep].numpy(), c[keep].numpy()

test_strip = sorted((RAW / "ai4shipwrecks" / "test" / "images").glob("*.png"))[5]
gray = load_gray(test_strip)
t0 = time.time()
models = {}
if TRAIN_DETECTOR: models["detector"] = YOLO(str(DET_BEST))
if TRAIN_SEGMENTER: models["segmenter"] = YOLO(str(SEG_BEST))
vis = cv2.cvtColor(gray, cv2.COLOR_GRAY2RGB)
for tag, m in models.items():
    b, s, c = sliced_predict(m, gray)
    col = (255, 60, 60) if tag == "detector" else (60, 255, 60)
    for (x1, y1, x2, y2), sc, cl in zip(b.astype(int), s, c):
        cv2.rectangle(vis, (x1, y1), (x2, y2), col, 6)
        cv2.putText(vis, f"{m.names[int(cl)]} {sc:.2f}", (x1, max(30, y1 - 10)), cv2.FONT_HERSHEY_SIMPLEX, 1.5, col, 3)
    print(f"{tag}: {len(b)} detections")
print(f"{test_strip.name} {gray.shape} processed in {time.time() - t0:.1f}s")
gt = load_gray(test_strip.parent.parent / "labels" / test_strip.name) > 0
fig, ax = plt.subplots(1, 2, figsize=(14, 14 * gray.shape[0] / gray.shape[1] / 2 + 1))
ax[0].imshow(gray, cmap="gray"); ax[0].contour(gt, colors="yellow"); ax[0].set_title("Ground truth (yellow)")
ax[1].imshow(vis); ax[1].set_title("Red: detector | Green: segmenter")
for a in ax: a.axis("off")
plt.show()

## 9. Export for the FastAPI service and edge devices
* `best.pt`: PyTorch weights (FastAPI on GPU/CPU)
* `best.onnx`: ONNX (CPU / onnxruntime / edge devices such as Jetson or Raspberry Pi)
* `model_card.json`: classes, hazard flags, calibrated thresholds, preprocessing, tiling, and test metrics. This file is the contract between the models and the API.

In [ ]:
def cpu_latency_ms(onnx_path, n=20):
    m = YOLO(str(onnx_path), task="segment" if "seg" in str(onnx_path) else "detect")
    dummy = np.random.randint(0, 255, (IMGSZ, IMGSZ, 3), np.uint8)
    m.predict(dummy, imgsz=IMGSZ, device="cpu", verbose=False)
    t = time.time()
    for _ in range(n):
        m.predict(dummy, imgsz=IMGSZ, device="cpu", verbose=False)
    return round((time.time() - t) / n * 1000, 1)

EXPORT = CONTENT / "sonar_models"
shutil.rmtree(EXPORT, ignore_errors=True)
jobs = []
if TRAIN_DETECTOR: jobs.append((DET_NAME, DET_BEST, "detect", DET_WEIGHTS, CLASSES))
if TRAIN_SEGMENTER: jobs.append((SEG_NAME, SEG_BEST, "segment", SEG_WEIGHTS, ["shipwreck"]))

for name, best, task, base, names in jobs:
    out = EXPORT / name
    out.mkdir(parents=True)
    onnx_path = Path(YOLO(str(best)).export(format="onnx", imgsz=IMGSZ, simplify=True, opset=12))
    shutil.copy(best, out / "best.pt")
    shutil.copy(onnx_path, out / "best.onnx")
    run = RUNS_DIR / name
    for f in ["results.csv", "results.png", "args.yaml"]:
        if (run / f).exists(): shutil.copy(run / f, out / f)
    test_dir = RUNS_DIR / f"{name}_test"
    if test_dir.exists():
        shutil.copytree(test_dir, out / "test_eval", dirs_exist_ok=True)
    metrics, thresholds, speed = RESULTS[name]
    card = {
        "name": name, "task": task, "architecture": base.replace(".pt", ""),
        "classes": names, "hazard": {n: HAZARD.get(n, True) for n in names},
        "input": {"imgsz": IMGSZ, "color": "grayscale replicated to 3 channels (BGR)"},
        "tiling": {"tile": TILE, "overlap": TILE_OVERLAP, "nms_iou": 0.5},
        "thresholds": thresholds, "default_conf": 0.25,
        "test_metrics": metrics, "train_device_speed_ms": speed,
        "cpu_onnx_latency_ms_per_tile": cpu_latency_ms(out / "best.onnx"),
        "datasets": json.load(open(WORK / "dataset_stats.json"))["det" if task == "detect" else "seg"],
        "trained_at": time.strftime("%Y-%m-%d %H:%M:%S"), "epochs": EPOCHS, "quick_test": QUICK_TEST,
    }
    (out / "model_card.json").write_text(json.dumps(card, indent=2, default=float))
    print(f"{name}: exported, CPU latency {card['cpu_onnx_latency_ms_per_tile']} ms/tile")

archive = shutil.make_archive(str(CONTENT / "sonar_models"), "zip", EXPORT)
shutil.copy(archive, RUNS_DIR / "sonar_models.zip")
print("Saved:", archive, "and", RUNS_DIR / "sonar_models.zip")

## 10. Download

In [ ]:
if IN_COLAB:
    from google.colab import files
    files.download(str(CONTENT / "sonar_models.zip"))